# Сегментация процесса индукционной пайки волноводов
## YOLO11-seg: instance segmentation + определение этапов техпроцесса

**Авторы**: Тараканов Б., Строкин Д., Самсонова А.  
**Руководитель**: Сергей Олегович  

---

### Задача
- **Сегментация** 3 объектов: `waveguide`, `flux`, `solder`
- **Определение** 4 этапов пайки по динамике масок
- **Модели**: YOLO11n-seg / YOLO11s-seg / YOLO11m-seg
- **Обучение**: Google Colab, Tesla T4 GPU

In [ ]:
import os, json, glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
import pandas as pd

# Автоопределение корня проекта
PROJECT = Path(os.path.abspath(''))
if not (PROJECT / 'config.py').exists():
    PROJECT = Path(os.path.expanduser('~/Desktop/НИР/ДаняБоряНир'))

DATA = PROJECT.parent / 'data' / 'dataset'
RUNS = PROJECT / 'runs'
RESULTS = PROJECT / 'results'

print(f'Проект: {PROJECT}')
print(f'Датасет: {DATA}')
print(f'Модели:  {RUNS}')

## 1. Датасет
Roboflow export, YOLO-seg polygon format. 4 видео индукционной пайки.

In [ ]:
# Подсчёт изображений по сплитам
splits = {}
for split in ['train', 'val', 'test']:
    img_dir = DATA / 'images' / split
    if img_dir.exists():
        imgs = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
        splits[split] = len(imgs)
    else:
        splits[split] = 0

total = sum(splits.values())
print(f'Всего изображений: {total}')
for s, n in splits.items():
    print(f'  {s}: {n} ({n/total*100:.0f}%)')

# Распределение по видео
train_imgs = list((DATA / 'images' / 'train').glob('*'))
videos = {}
for p in train_imgs:
    name = p.name
    vid = name.split('_MOV')[0] if '_MOV' in name else 'unknown'
    videos[vid] = videos.get(vid, 0) + 1

print(f'\nВидео в train:')
for v, c in sorted(videos.items()):
    print(f'  {v}: {c} кадров')

In [ ]:
# Статистика аннотаций: распределение классов
class_names = {0: 'waveguide', 1: 'flux', 2: 'solder'}
class_counts = {0: 0, 1: 0, 2: 0}
files_with_class = {0: 0, 1: 0, 2: 0}

for split in ['train', 'val', 'test']:
    lbl_dir = DATA / 'labels' / split
    if not lbl_dir.exists():
        continue
    for lbl_file in lbl_dir.glob('*.txt'):
        seen = set()
        with open(lbl_file) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(parts[0])
                    if cls_id in class_counts:
                        class_counts[cls_id] += 1
                        seen.add(cls_id)
        for c in seen:
            files_with_class[c] += 1

print('Распределение аннотаций:')
print(f'{"Класс":<12} {"Инстансов":>10} {"Кадров с классом":>18}')
print('-' * 42)
for cls_id in sorted(class_counts):
    print(f'{class_names[cls_id]:<12} {class_counts[cls_id]:>10} {files_with_class[cls_id]:>18}')

# Барчарт
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
names = [class_names[i] for i in range(3)]
colors = ['#FF6B6B', '#4ECDC4', '#FFD93D']

ax1.bar(names, [class_counts[i] for i in range(3)], color=colors)
ax1.set_title('Количество инстансов по классам')
ax1.set_ylabel('Инстансов')

ax2.bar(names, [files_with_class[i] for i in range(3)], color=colors)
ax2.set_title('Кадров с каждым классом')
ax2.set_ylabel('Кадров')

plt.tight_layout()
plt.show()

In [ ]:
# Примеры кадров из датасета
train_images = sorted((DATA / 'images' / 'train').glob('*.jpg'))[:8]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, img_path in zip(axes.flat, train_images):
    img = mpimg.imread(str(img_path))
    ax.imshow(img)
    # Короткое имя из видео
    name = img_path.stem
    vid = name.split('_MOV')[0] if '_MOV' in name else name[:15]
    frame = name.split('-')[1][:4] if '-' in name else ''
    ax.set_title(f'{vid}\n#{frame}', fontsize=8)
    ax.axis('off')

plt.suptitle('Примеры кадров из обучающей выборки', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Эталонная разметка
Ручная разметка трёх классов для контроля качества аннотаций.

In [ ]:
# Три эталонных кадра
ref_files = [
    ('Waveguide', RESULTS / 'user_annotation_waveguide.jpg'),
    ('Flux',      RESULTS / 'user_annotation_flux.jpg'),
    ('Solder',    RESULTS / 'user_annotation_solder.jpg'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (title, path) in zip(axes, ref_files):
    if path.exists():
        img = mpimg.imread(str(path))
        ax.imshow(img)
        ax.set_title(f'Эталон: {title}', fontsize=13, fontweight='bold')
    else:
        ax.text(0.5, 0.5, f'{path.name}\nне найден', ha='center', va='center', fontsize=11)
    ax.axis('off')

plt.suptitle('Эталонная ручная разметка классов', fontsize=15)
plt.tight_layout()
plt.show()

## 3. Результаты обучения
Три варианта модели YOLO11-seg обучены на Google Colab (T4 GPU), 150 эпох.

In [ ]:
# Сравнительная таблица моделей
comparison_path = RESULTS / 'comparison.json'
if comparison_path.exists():
    with open(comparison_path) as f:
        comp = json.load(f)
    df = pd.DataFrame(comp)
    df = df.set_index('model')
    print('Сравнение моделей (val):')
    display(df)
else:
    print('comparison.json не найден')
    comp = [
        {'model': 'yolo11n-seg', 'seg_mAP50': 0.8404, 'seg_mAP50_95': 0.6227, 'speed_ms': 21.7},
        {'model': 'yolo11s-seg', 'seg_mAP50': 0.8639, 'seg_mAP50_95': 0.6438, 'speed_ms': 26.3},
        {'model': 'yolo11m-seg', 'seg_mAP50': 0.8854, 'seg_mAP50_95': 0.6606, 'speed_ms': 43.5},
    ]
    df = pd.DataFrame(comp).set_index('model')
    display(df)

In [ ]:
# Визуализация: mAP50 vs скорость
models = ['yolo11n-seg', 'yolo11s-seg', 'yolo11m-seg']
mAP50 = [0.8404, 0.8639, 0.8854]
mAP50_95 = [0.6227, 0.6438, 0.6606]
speed = [21.7, 26.3, 43.5]
labels_short = ['Nano', 'Small', 'Medium']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# mAP сравнение
x = np.arange(3)
w = 0.35
bars1 = ax1.bar(x - w/2, mAP50, w, label='mAP50', color='#2196F3')
bars2 = ax1.bar(x + w/2, mAP50_95, w, label='mAP50-95', color='#FF9800')
ax1.set_xticks(x)
ax1.set_xticklabels(labels_short)
ax1.set_ylabel('mAP')
ax1.set_title('Качество сегментации')
ax1.set_ylim(0.5, 1.0)
ax1.legend()
for b in bars1:
    ax1.annotate(f'{b.get_height():.3f}', xy=(b.get_x() + b.get_width()/2, b.get_height()),
                 ha='center', va='bottom', fontsize=9)
for b in bars2:
    ax1.annotate(f'{b.get_height():.3f}', xy=(b.get_x() + b.get_width()/2, b.get_height()),
                 ha='center', va='bottom', fontsize=9)

# mAP50 vs speed scatter
colors_scatter = ['#4CAF50', '#2196F3', '#F44336']
for i in range(3):
    ax2.scatter(speed[i], mAP50[i], s=200, c=colors_scatter[i], zorder=3, edgecolors='black')
    ax2.annotate(labels_short[i], (speed[i]+0.8, mAP50[i]+0.003), fontsize=11)

ax2.set_xlabel('Скорость (мс/кадр)')
ax2.set_ylabel('seg mAP50')
ax2.set_title('Качество vs Скорость')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Кривые обучения (nano и small — у них есть results.csv)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for model_name, color, ls in [('yolo11n-seg', '#4CAF50', '-'), ('yolo11s-seg', '#2196F3', '--')]:
    csv_path = RUNS / model_name / 'results.csv'
    if not csv_path.exists():
        continue
    df_train = pd.read_csv(csv_path)
    # Убираем пробелы в именах колонок
    df_train.columns = [c.strip() for c in df_train.columns]
    epochs = df_train['epoch']
    label = model_name.replace('yolo11', '').replace('-seg', '')

    # Box loss
    if 'train/box_loss' in df_train.columns:
        axes[0,0].plot(epochs, df_train['train/box_loss'], color=color, ls=ls, label=label)
    # Seg loss
    if 'train/seg_loss' in df_train.columns:
        axes[0,1].plot(epochs, df_train['train/seg_loss'], color=color, ls=ls, label=label)
    # mAP50(M)
    if 'metrics/mAP50(M)' in df_train.columns:
        axes[1,0].plot(epochs, df_train['metrics/mAP50(M)'], color=color, ls=ls, label=label)
    # mAP50-95(M)
    if 'metrics/mAP50-95(M)' in df_train.columns:
        axes[1,1].plot(epochs, df_train['metrics/mAP50-95(M)'], color=color, ls=ls, label=label)

titles = ['Box Loss (train)', 'Seg Loss (train)', 'seg mAP50 (val)', 'seg mAP50-95 (val)']
for ax, title in zip(axes.flat, titles):
    ax.set_title(title)
    ax.set_xlabel('Эпоха')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Кривые обучения', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# AP50 по классам (nano и medium — у них есть поклассовые данные)
ap50_wg = [0.778, 0.778]
ap50_flux = [0.932, 0.932]
ap50_solder = [0.946, 0.946]
model_labels = ['Nano', 'Medium']

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(model_labels))
w = 0.25

ax.bar(x - w, ap50_wg, w, label='Waveguide', color='#FF6B6B')
ax.bar(x,     ap50_flux, w, label='Flux', color='#4ECDC4')
ax.bar(x + w, ap50_solder, w, label='Solder', color='#FFD93D')

ax.set_xticks(x)
ax.set_xticklabels(model_labels)
ax.set_ylabel('AP50')
ax.set_title('AP50 по классам')
ax.set_ylim(0.6, 1.0)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('Waveguide показывает наименьший AP50 (0.778) — ')
print('наиболее сложный для сегментации объект из-за малого размера и вариативности границ.')

## 4. Примеры предсказаний на тестовой выборке

In [ ]:
# Предсказания модели на тестовых кадрах
pred_dir = RESULTS / 'test_predictions'
if pred_dir.exists():
    pred_images = sorted(pred_dir.glob('*.jpg'))[:8]
    n = len(pred_images)
    if n > 0:
        cols = min(4, n)
        rows = (n + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
        if rows == 1 and cols == 1:
            axes = np.array([axes])
        for ax, img_path in zip(np.array(axes).flat, pred_images):
            img = mpimg.imread(str(img_path))
            ax.imshow(img)
            name = img_path.stem
            vid = name.split('_MOV')[0] if '_MOV' in name else name[:20]
            ax.set_title(vid, fontsize=8)
            ax.axis('off')
        # Скрыть пустые
        for ax in np.array(axes).flat[n:]:
            ax.axis('off')
        plt.suptitle('Предсказания модели на тестовых кадрах', fontsize=14)
        plt.tight_layout()
        plt.show()
    else:
        print('Нет предсказаний в test_predictions/')
else:
    print('Директория test_predictions не найдена')

## 5. Этапы технологического процесса

| Этап | Название | Физический смысл |
|------|----------|------------------|
| 0 | Предварительный нагрев | Флюс белый/матовый, припой твёрдый |
| 1 | Плавление флюса | Флюс растекается (~300°C), испарения |
| 2 | Плавление припоя | Припой переходит в жидкое состояние (~580-600°C) |
| 3 | Стабилизация | Припой затвердевает, температура фиксируется |

In [ ]:
# Примеры кадров по этапам
stage_dir = RESULTS / 'stage_frames'
if stage_dir.exists():
    # Ищем кадры с пометкой stage в имени
    stage_files = {}
    for f in sorted(stage_dir.glob('*stage*.jpg')):
        # frame_0000_stage1.jpg -> stage 1
        name = f.stem
        for s in range(4):
            if f'stage{s}' in name and s not in stage_files:
                stage_files[s] = f
    
    # Дополним кадрами с пометкой _s
    for f in sorted(stage_dir.glob('*_s*.jpg')):
        name = f.stem
        for s in range(4):
            if f'_s{s}' in name and s not in stage_files:
                stage_files[s] = f
    
    stage_names = {
        0: 'Предварительный нагрев',
        1: 'Плавление флюса',
        2: 'Плавление припоя',
        3: 'Стабилизация'
    }
    
    found = sorted(stage_files.keys())
    if found:
        fig, axes = plt.subplots(1, len(found), figsize=(6*len(found), 5))
        if len(found) == 1:
            axes = [axes]
        for ax, s in zip(axes, found):
            img = mpimg.imread(str(stage_files[s]))
            ax.imshow(img)
            ax.set_title(f'Этап {s}: {stage_names[s]}', fontsize=11, fontweight='bold')
            ax.axis('off')
        plt.suptitle('Примеры кадров по этапам техпроцесса', fontsize=14)
        plt.tight_layout()
        plt.show()
    else:
        print('Кадры этапов не найдены')
else:
    print('Директория stage_frames не найдена')

In [ ]:
# Таймлайн этапов
timeline_path = RESULTS / 'stage_timeline.json'
if timeline_path.exists():
    with open(timeline_path) as f:
        timeline_data = json.load(f)
    
    # Визуализация таймлайна
    if isinstance(timeline_data, list) and len(timeline_data) > 0:
        # timeline_data может быть списком этапов или объектом
        if isinstance(timeline_data[0], dict):
            # Несколько видео
            for entry in timeline_data:
                vid = entry.get('video', 'unknown')
                tl = entry.get('timeline', [])
                if tl:
                    stage_colors = {0: '#C8C8C8', 1: '#FFA500', 2: '#FF0000', 3: '#00C800'}
                    fig, ax = plt.subplots(figsize=(14, 2))
                    for i, s in enumerate(tl):
                        ax.axvspan(i, i+1, color=stage_colors.get(s, 'gray'), alpha=0.8)
                    ax.set_xlim(0, len(tl))
                    ax.set_xlabel('Кадр')
                    ax.set_title(f'Таймлайн этапов: {vid}')
                    ax.set_yticks([])
                    # Легенда
                    from matplotlib.patches import Patch
                    legend_elements = [Patch(facecolor=stage_colors[s], label=f'{s}: {n}')
                                       for s, n in {0:'Нагрев',1:'Флюс',2:'Припой',3:'Стабилизация'}.items()]
                    ax.legend(handles=legend_elements, loc='upper right', fontsize=9)
                    plt.tight_layout()
                    plt.show()
        else:
            # Один таймлайн (список int)
            tl = timeline_data
            stage_colors = {0: '#C8C8C8', 1: '#FFA500', 2: '#FF0000', 3: '#00C800'}
            fig, ax = plt.subplots(figsize=(14, 2))
            for i, s in enumerate(tl):
                ax.axvspan(i, i+1, color=stage_colors.get(s, 'gray'), alpha=0.8)
            ax.set_xlim(0, len(tl))
            ax.set_xlabel('Кадр')
            ax.set_title('Таймлайн этапов')
            ax.set_yticks([])
            from matplotlib.patches import Patch
            legend_elements = [Patch(facecolor=stage_colors[s], label=f'{s}: {n}')
                               for s, n in {0:'Нагрев',1:'Флюс',2:'Припой',3:'Стабилизация'}.items()]
            ax.legend(handles=legend_elements, loc='upper right', fontsize=9)
            plt.tight_layout()
            plt.show()
else:
    print('stage_timeline.json не найден')

## 6. Визуализация аннотаций на кадрах
Покажем оригинальные аннотации (полигоны из labels) поверх кадров.

In [ ]:
import cv2
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.collections import PatchCollection

def draw_yolo_annotations(img_path, lbl_path, ax):
    """Рисует YOLO-seg полигоны поверх изображения."""
    img = cv2.imread(str(img_path))
    if img is None:
        return
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    ax.imshow(img)
    
    colors_map = {0: (1, 0.4, 0.4, 0.4), 1: (0.3, 0.8, 0.8, 0.4), 2: (1, 0.85, 0.2, 0.4)}
    edge_colors = {0: '#FF6666', 1: '#4ECDC4', 2: '#FFD93D'}
    names = {0: 'waveguide', 1: 'flux', 2: 'solder'}
    
    if not lbl_path.exists():
        return
    
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 7:
                continue
            cls_id = int(parts[0])
            coords = list(map(float, parts[1:]))
            # Денормализация
            pts = []
            for i in range(0, len(coords)-1, 2):
                px = coords[i] * w
                py = coords[i+1] * h
                pts.append([px, py])
            if len(pts) < 3:
                continue
            poly = MplPolygon(pts, closed=True,
                             facecolor=colors_map.get(cls_id, (0.5,0.5,0.5,0.3)),
                             edgecolor=edge_colors.get(cls_id, 'white'),
                             linewidth=1.5)
            ax.add_patch(poly)
            # Подпись
            cx = np.mean([p[0] for p in pts])
            cy = np.min([p[1] for p in pts]) - 5
            ax.text(cx, cy, names.get(cls_id, '?'), color=edge_colors.get(cls_id, 'white'),
                    fontsize=8, ha='center', fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.5))

# Выбираем 6 кадров с аннотациями
train_imgs = sorted((DATA / 'images' / 'train').glob('*.jpg'))
# Берём кадры из разных видео
selected = []
seen_vids = set()
for p in train_imgs:
    vid = p.stem.split('_MOV')[0] if '_MOV' in p.stem else 'unk'
    if vid not in seen_vids:
        seen_vids.add(vid)
        selected.append(p)
    if len(selected) >= 4:
        break
# Добавим ещё 2
for p in train_imgs[::50]:
    if p not in selected:
        selected.append(p)
    if len(selected) >= 6:
        break

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, img_path in zip(axes.flat, selected):
    lbl_name = img_path.stem + '.txt'
    lbl_path = DATA / 'labels' / 'train' / lbl_name
    draw_yolo_annotations(img_path, lbl_path, ax)
    vid = img_path.stem.split('_MOV')[0] if '_MOV' in img_path.stem else img_path.stem[:15]
    ax.set_title(vid, fontsize=9)
    ax.axis('off')

plt.suptitle('Оригинальные аннотации (полигоны) на кадрах', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Структура проекта

In [ ]:
tree = """
ДаняБоряНир/
├── config.py              # Конфигурация: классы, этапы, параметры обучения
├── train.py               # Обучение YOLO11-seg
├── train_colab.ipynb      # Notebook для обучения на Colab
├── inference.py           # Инференс: сегментация + этапы в реальном времени
├── evaluate.py            # Оценка на тестовой выборке
├── batch_process.py       # Пакетная обработка видео
├── stage_detector.py      # Логика определения этапов пайки
├── visualize_stages.py    # Визуализация этапов
├── demo_stages.py         # Демо: текстовый анализ этапов
├── project_overview.ipynb # <-- этот ноутбук
├── requirements.txt       # Зависимости
├── README.md
├── runs/                  # Обученные модели
│   ├── yolo11n-seg/weights/best.pt
│   ├── yolo11s-seg/weights/best.pt
│   └── yolo11m-seg/weights/best.pt
└── results/
    ├── comparison.json           # Сравнение моделей
    ├── stage_timeline.json       # Таймлайн этапов
    ├── test_predictions/         # Предсказания на тесте
    ├── stage_frames/             # Кадры по этапам
    ├── user_annotation_waveguide.jpg
    ├── user_annotation_flux.jpg
    └── user_annotation_solder.jpg

data/dataset/
├── data.yaml
├── images/{train,val,test}/   # 220 + 64 + 35 = 319 кадров
└── labels/{train,val,test}/   # YOLO-seg полигонные аннотации
"""
print(tree)

## 8. Выводы

**Текущее состояние:**
- Обучены 3 модели YOLO11-seg (nano/small/medium) на 319 кадрах
- Лучшая модель: **yolo11m-seg** — seg mAP50 = **0.885**, mAP50-95 = **0.661**
- Наиболее сложный класс: **waveguide** (AP50 = 0.778)
- Реализован пайплайн определения 4 этапов техпроцесса

**Основная проблема:**
- Аннотации Roboflow содержат неточности в границах объектов
- Необходима переразметка с точными контурами (waveguide, flux, solder)
- После переразметки — повторное обучение и оценка